#Imports

In [1]:
!pip install mltk

  Preparing metadata (setup.py) ... done
  Created wheel for mltk: filename=mltk-0.0.5-py3-none-any.whl size=7770 sha256=ace49d0b815d0bc4b2ea87ca261e6778a9c221556df070c90dd2d88a3334da99
  Stored in directory: /root/.cache/pip/wheels/c8/86/11/f407a4634aa2cbea87662ff387bd8aba39d2946f254e185c85
Successfully built mltk


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import word_tokenize
import nltk

#Data loading

In [3]:
with open('DATA.txt', 'r', encoding='utf-8') as f:
    text = f.read()

#Data Preprocessing

##Basic pre-processing

In [4]:
# #remove line breaks
# text = text.replace('\n', ' ')
# text = text.replace('\r', ' ')

In [5]:
# #remove extra space
# import re

# text = re.sub(r'\s+', ' ', text).strip()

##Tokenization

In [6]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [7]:
tokens = word_tokenize(text.lower())

#Vocabulary building

In [8]:
vocab = {"<unk>":0}

for tokens in Counter(tokens).keys():
  if tokens not in vocab.keys():
    vocab[tokens] = len(vocab)


In [9]:
print(f"Number of words in vocabulary = {len(vocab)}")

Number of words in vocabulary = 9398


#Indexing each word of a sentance

In [10]:
input_sentance = text.split('\n')

In [11]:
def text_to_indices(sentance , vocab):
  num_sentance = []
  for token in sentance:
    if token in vocab:
      num_sentance.append(vocab[token])
    else:
      num_sentance.append(vocab["<unk>"])
  return num_sentance

In [12]:
input_numerical_sentences = []

for sentence in input_sentance:
  input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))

##Defining the training sequance

In [13]:
training_sequence = []
for sentence in input_numerical_sentences:

  for i in range(1, len(sentence)):
    training_sequence.append(sentence[:i+1])

In [14]:
training_sequence[:5]

[[1, 2], [1, 2, 3], [1, 2, 3, 4], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]]

#Padding

In [15]:
len_list = []

for sequence in training_sequence:
  len_list.append(len(sequence))

max(len_list)

29

In [16]:
padded_training_sequence = []
for sequence in training_sequence:

  padded_training_sequence.append([0]*(max(len_list) - len(sequence)) + sequence)

In [17]:
#Converting to tensor
padded_training_sequence = torch.tensor(padded_training_sequence , dtype=torch.long)

##Splitting into X and y

In [18]:
X = padded_training_sequence[: , :-1]
y = padded_training_sequence[: , -1]

#Dataset and DataLoader

In [19]:
class CustomDataset(Dataset):

  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, index):
    return self.X[index] , self.y[index]

In [20]:
dataset = CustomDataset(X , y)

In [21]:
dataloader = DataLoader(dataset , batch_size=64 , shuffle=True)

#LSTM Archetecture

In [22]:
class LSTMModel(nn.Module):

  def __init__(self , vocab_size) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size , 100)
    self.lstm = nn.LSTM(100 , 150 , batch_first=True)
    self.linear = nn.Linear(150 , vocab_size)

  def forward(self , x):
    embedded = self.embedding(x)
    intermediate_hidden_states , (final_hidden_state , final_cell_state) = self.lstm(embedded)
    output = self.linear(final_hidden_state.squeeze(0))
    return output

#Training the LSTM Model

In [23]:
model = LSTMModel(len(vocab))

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [25]:
model.to(device)

LSTMModel(
  (embedding): Embedding(9398, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (linear): Linear(in_features=150, out_features=9398, bias=True)
)

In [26]:
epochs = 100
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [27]:
# training loop

for epoch in range(epochs):
  total_loss = 0

  for batch_x, batch_y in dataloader:

    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_x)

    loss = criterion(output, batch_y)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch + 1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 10565.6240
Epoch: 2, Loss: 9050.0562
Epoch: 3, Loss: 8334.7765
Epoch: 4, Loss: 7744.0882
Epoch: 5, Loss: 7212.7131
Epoch: 6, Loss: 6728.9307
Epoch: 7, Loss: 6289.5694
Epoch: 8, Loss: 5896.1048
Epoch: 9, Loss: 5535.2945
Epoch: 10, Loss: 5209.8117
Epoch: 11, Loss: 4909.0212
Epoch: 12, Loss: 4635.7373
Epoch: 13, Loss: 4382.4537
Epoch: 14, Loss: 4149.2403
Epoch: 15, Loss: 3936.7561
Epoch: 16, Loss: 3738.3204
Epoch: 17, Loss: 3557.2187
Epoch: 18, Loss: 3388.5526
Epoch: 19, Loss: 3234.2851
Epoch: 20, Loss: 3089.0688
Epoch: 21, Loss: 2955.5767
Epoch: 22, Loss: 2831.9310
Epoch: 23, Loss: 2717.3162
Epoch: 24, Loss: 2613.6209
Epoch: 25, Loss: 2511.5595
Epoch: 26, Loss: 2420.0535
Epoch: 27, Loss: 2331.1253
Epoch: 28, Loss: 2254.8610
Epoch: 29, Loss: 2177.0577
Epoch: 30, Loss: 2103.1265
Epoch: 31, Loss: 2042.0856
Epoch: 32, Loss: 1977.6733
Epoch: 33, Loss: 1920.7827
Epoch: 34, Loss: 1865.6309
Epoch: 35, Loss: 1814.5746
Epoch: 36, Loss: 1766.0029
Epoch: 37, Loss: 1719.4324
Epoch: 38

#Prediction

In [30]:
def prediction(model, vocab, text):

  # tokenize
  tokenized_text = word_tokenize(text.lower())

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  # padding
  padded_text = torch.tensor([0] * (61 - len(numerical_text)) + numerical_text, dtype=torch.long).unsqueeze(0)

  # Move the input tensor to the same device as the model
  padded_text = padded_text.to(device)

  # send to model
  output = model(padded_text)

  # predicted index
  value, index = torch.max(output, dim=1)

  # merge with text
  return text + " " + list(vocab.keys())[index]

In [32]:
prediction(model, vocab, "I was ")

'I was  informed'